# XLS-R Nepali Baseline Audit

Evaluates `gagan3012/wav2vec2-xlsr-nepali` on OpenSLR-43 (`gauravparajuli/slr43`)
-- the same corpus it was originally trained/self-evaluated on -- to sanity-check
its published 5.97% WER claim. Single model, single dataset, single purpose.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import sys
sys.path.insert(0, '.')
from src.xlsr_baseline import (
    load_xlsr_model_and_processor,
    load_openslr43_test_dataset,
    resample_if_needed,
    transcribe_ctc,
    build_openslr43_manifest_rows,
    write_openslr43_manifest,
    SELF_REPORTED_WER,
)
from src.wer_metrics import compute_wer_cer

model, processor = load_xlsr_model_and_processor()
device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
model.to(device)
print('device:', device)

In [ ]:
N_SAMPLES = 150  # set to None for the full 2,064-utterance corpus
dataset = load_openslr43_test_dataset(max_samples=N_SAMPLES)
rows = build_openslr43_manifest_rows(dataset)
write_openslr43_manifest(rows, 'data/manifests/xlsr_openslr43_test_manifest.csv')
len(dataset)

In [ ]:
references, hypotheses = [], []
for i, row in enumerate(dataset):
    audio = row['audio']
    speech = resample_if_needed(audio['array'], audio['sampling_rate'])
    hyp = transcribe_ctc(model, processor, speech, device)
    references.append(rows[i]['text'])
    hypotheses.append(hyp)
    if (i + 1) % 25 == 0:
        print(f'{i + 1}/{len(dataset)} done')

wer, cer = compute_wer_cer(references, hypotheses)
print(f'WER={wer:.4f} CER={cer:.4f} on {len(dataset)} utterances')
print(f'Self-reported WER: {SELF_REPORTED_WER:.4f}')

In [ ]:
import json
results = {
    'model': 'gagan3012/wav2vec2-xlsr-nepali',
    'dataset': 'OpenSLR-43 (gauravparajuli/slr43)',
    'num_utterances': len(dataset),
    'wer': wer,
    'cer': cer,
    'self_reported_wer': SELF_REPORTED_WER,
}
with open('reports/xlsr_baseline_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print('saved reports/xlsr_baseline_results.json')